# Conversion des fichiers en vecteurs intéressants avec librosa

In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram
import plotly.graph_objects as go
import os
import warnings
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [62]:
def verbose(message,verb,importance = 0):
    """
    Print the message if verbose is set to True.
    """
    if verb>importance:
        print(message)

In [63]:
# Variables globales
li_notes = ['time','A','A#','B','C','C#','D','D#','E','F','F#','G','G#']
N=3
loudness_resolution = 50
overlap = 0.5

## Fonctions de preprocessing

In [64]:
def get_dataframes(chroma_f, verb=0):
    """Renvoie un dictionaire sous la forme titre : dataframe des chroma features"""
    li_notes = ['time','A','A#','B','C','C#','D','D#','E','F','F#','G','G#']
    df = pd.DataFrame(chroma_f.T, columns=li_notes[1:])
    while (df.tail(1).values == 0).all():
        df = df.iloc[:-1]
    return df

In [65]:
def quantize(dataframe, loudness_resolution, verb):
    """Quantifie les valeurs de chroma features entre 0 et loudness_resolution"""
    dataframe = dataframe.to_numpy()
    dataframe/= np.max(dataframe)
    dataframe = np.ceil(dataframe * loudness_resolution)
    verbose(f"Tableau quantifié : {dataframe}", verb, 0)
    return dataframe

In [66]:
def get_frames(dataframe,N,overlap, verb):
    """Découpe le dataframe en 2**N frames avec une proportion overlap de recoupement"""
    step = int((1-overlap)*len(dataframe)/2**N)
    frames = []
    for i in range(2**N):
        frames.append(dataframe[i*step:(i+1)*step,:])
    verbose(f"Nouvelles frames : {frames}", verb, 0)
    return frames

In [67]:
def histogram(frame,loudness_resolution,verb, affiche = False):
    """Renvoie l'histogramme de la frame"""
    step = frame.shape[0]
    histogram = np.zeros((loudness_resolution+1, 12))
    for i in range(step):
        for j in range(12):
            loudness_level = int(frame[i, j])
            for k in range(loudness_level+1):
                histogram[k, j] += 1
    histogram /= np.max(histogram)
    if affiche:
        plt.figure(figsize=(5,10))
        sns.heatmap(histogram, cmap='coolwarm', cbar=True, xticklabels=li_notes[1:], yticklabels=np.arange(loudness_resolution+1))
        plt.title('Histogramme de la frame')
        plt.xlabel('Chroma Features')
        plt.ylabel('Loudness Level')
        plt.show()
    return histogram

In [68]:
def dico_hist(dico,loudness_resolution=50,N=3,overlap=0.5,verb=0):
    res = {}
    for titre, chroma in dico.items():
        dataframes = get_dataframes(chroma, verb)
        data = quantize(dataframes, loudness_resolution, verb)
        frames = get_frames(data, N, overlap, verb)
        histograms = []
        for frame in frames:
            histograms.append(histogram(frame, loudness_resolution, verb))
        res[titre] = histograms
    verbose(f"Dictionnaire d'histogrammes : {res}", verb, 0)
    return res

## Découpage

In [ ]:
import os
import librosa
chroma_dict = {}

audio_dir = "audiofiles\csv"

for filename in os.listdir(audio_dir):
    if filename.endswith(".csv"):
        file_path = os.path.join(audio_dir, filename)
        df_csv = pd.read_csv(file_path)
        cols = df_csv.columns[1:]
        print(df_csv.head())
        while (df_csv[cols].iloc[0] == 0).all():
            df_csv = df_csv.iloc[1:].reset_index(drop=True)
        while (df_csv[cols].iloc[-1] == 0).all():
            df_csv = df_csv.iloc[:-1].reset_index(drop=True)
        chroma = df_csv.iloc[:, 1:].values.T
        print(f"Tableau de {filename} : {df_csv.head()}")
        print(f"Matrice de {filename} : {chroma}")
        chroma_dict[filename] = chroma
dico = dico_hist(chroma_dict, loudness_resolution=loudness_resolution, N=N, overlap=overlap, verb=0)
with open("dico_hist_supp2.pkl", "wb") as f:
    pickle.dump(dico, f)

<>:5: SyntaxWarning:

invalid escape sequence '\c'

<>:5: SyntaxWarning:

invalid escape sequence '\c'

C:\Users\PCAJM\AppData\Local\Temp\ipykernel_18220\3646269495.py:5: SyntaxWarning:

invalid escape sequence '\c'



   TIME    A   Bb    B    C   C#    D   Eb    E    F   F#    G   Ab
0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
1   0.1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
2   0.2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
3   0.3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
4   0.4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
Tableau de Beethoven_blackman-harris.csv :    TIME        A        Bb         B        C        C#         D        Eb  \
0   1.2  0.32492  0.330889  0.266153  1.57734  0.000000  0.432834  0.195493   
1   1.3  1.87926  0.002379  1.068100  1.44894  0.076973  0.448155  0.749649   
2   1.4  2.52737  0.090385  0.240883  1.62690  0.015554  0.244850  0.516279   
3   1.5  1.71180  0.001384  0.118072  2.71954  0.000000  0.862121  0.088432   
4   1.6  1.87129  0.000000  0.000000  2.17594  0.000000  0.014061  0.000000   

          E         F        F#         G        Ab  
0  0.732330  0.97111

## Test avec les autres éléments

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram
import plotly.graph_objects as go
import os
import warnings
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [71]:
li_filenames = []

for racine, _, fichiers in os.walk('cross-era_chroma-nnls'):
    for fichier in fichiers:
        chemin_relatif = os.path.relpath(os.path.join(racine, fichier))
        li_filenames.append(chemin_relatif)

In [72]:
def periode(str):
    try:
        date_deb = int(str[0:4])
        date_fin = int(str[5:9])
        date = (date_deb + date_fin)/2
        if date < 1500:
            return "Moyen-Âge"
        elif date < 1600:
            return "Renaissance"
        elif date < 1750:
            return "Baroque"
        elif date < 1800:
            return "Classique"
        elif date < 1880:
            return "Romantique"
        else:
            return "XXe siècle"
    except :
        return "Inconnu"

In [73]:
def dico_filenames(filename = "cross-era_annotations.csv"):
    dico = {}
    dico2 = {}
    df = pd.read_csv(filename, sep=',')
    for i in range(len(df)):
        dico[df['Filename'][i]] = df['Composer'][i]
        dico2[df['Filename'][i]] = periode(df['CompLifetime'][i])
    return dico, dico2

In [74]:
pd.read_csv('cross-era_annotations.csv', sep=',')

,Class,Filename,CrossEra-ID,Instrumentation,Key,Mode,Composer,CompLifetime,Country,Unnamed: 9
0,orchestra_baroque,CrossEra-0001_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0001,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
1,orchestra_baroque,CrossEra-0002_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0002,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
2,orchestra_baroque,CrossEra-0003_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0003,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
3,orchestra_baroque,CrossEra-0004_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0004,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
4,orchestra_baroque,CrossEra-0005_Albinoni_concerto_in_a_minor_bwv...,CrossEra-0005,orchestra,A,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
...,...,...,...,...,...,...,...,...,...,...
1995,piano_addon,CrossEra-1996_Weber_sonata_no._35_in_a_minor_o...,CrossEra-1996,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1996,piano_addon,CrossEra-1997_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1997,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1997,piano_addon,CrossEra-1998_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1998,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1998,piano_addon,CrossEra-1999_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1999,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN


In [75]:
dico_composers, dico_periodes = dico_filenames()
a = np.array(list(dico_composers.values()))
np.unique(a)
# Imprime toutes les clés ayant pour valeur 'major'
for key, value in dico_composers.items():
    if value == ' major':
        print(key)
dico_composers["CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3"] = 'Borodin; Alexander'
dico_composers["CrossEra-0673_Liszt_poems__mazeppa.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0674_Liszt_poems__prometheus.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3"] = 'Verdi; Giuseppe'
dico_composers["CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3"] = 'Cimarosa; Domenico'

CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3
CrossEra-0673_Liszt_poems__mazeppa.mp3
CrossEra-0674_Liszt_poems__prometheus.mp3
CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3
CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3


In [76]:
# Load the data
with open("hist_cross-era.pkl", "rb") as f:
    dico_cross_era = pickle.load(f)

In [77]:
# Load the data
with open("dico_hist_supp2.pkl", "rb") as f:
    dico_cross_era_sup = pickle.load(f)

In [78]:
X = []
etiquettes = []
y = []
periode = []
for file, histograms in dico_cross_era.items():
    composer = dico_composers[file.split('/')[1]]
    etiquettes.append(file)
    matrice = np.vstack(histograms)
    X.append(matrice.flatten())
    y.append(composer)
    periode.append(dico_periodes[file.split('/')[1]])
for file, histograms in dico_cross_era_sup.items():
    composer = file
    etiquettes.append(file)
    matrice = np.vstack(histograms)
    X.append(matrice.flatten())
    y.append(composer)
    periode.append(composer)


X = np.array(X)
X_scaled = StandardScaler().fit_transform(X)
y = np.array(y)
periodes = np.array(periode)
morceaux = np.array(etiquettes)

In [79]:
# Compter le nombre d'oeuvres par compositeur
oeuvres_par_compositeur = pd.Series(y).value_counts()
oeuvres_par_compositeur

Bach; Johann Sebastian           120
Mozart; Wolfgang Amadeus         114
Haydn; Joseph                    100
Shostakovich; Dmitri              82
Beethoven; Ludwig van             62
                                ... 
Beethoven_blackman-harris.csv      1
Beethoven_hamming.csv              1
Beethoven_gaussian.csv             1
Beethoven_blackman.csv             1
Beethoven_triangular.csv           1
Name: count, Length: 79, dtype: int64

In [80]:
# LDA 3D et visualisation Plotly
lda = LDA(n_components=3)
X_lda = lda.fit_transform(X_scaled, y)

# Construit un DataFrame pour l'affichage
df_lda = pd.DataFrame(X_lda, columns=['LD1', 'LD2', 'LD3'])
df_lda['composer'] = y
df_lda['morceau'] = morceaux

# Trace 3D avec Plotly Express
fig = px.scatter_3d(
    df_lda,
    x='LD1', y='LD2', z='LD3',
    color='composer',
    hover_data={'morceau': True, 'composer': True},
    title='Projection LDA 3D des morceaux par compositeur'
)
fig.update_traces(marker=dict(size=4, opacity=0.8))
fig.update_traces(marker=dict(size=4))
buttons = [
    dict(label='Tout masquer',
         method='restyle',
         args=['visible', ['legendonly'] * len(fig.data)]),
    
    dict(label='Tout afficher',
         method='restyle',
         args=['visible', [True] * len(fig.data)])
]

fig.update_layout(
    updatemenus=[dict(type='buttons', showactive=True, buttons=buttons)]
)
fig.show()